# Global aggregation and plots of the VAS CRU reconstruction (RGI 6.0 or RGI 7.0, all regions)

**What this notebook does**
1. Reads the regional result files (`regional_results_hist1901_2019.csv`) from the region folders (`vas_rgi60_regionXX` or `vas_rgi70_regionXX`, chosen with `RGI_VERSION` in the config cell) made by the reconstruction notebook.
2. Aggregates them into one global time series (sum of all regions) and saves all tables as CSV.
3. Plots volume, area and sea-level equivalent **per region**, **globally**, and **all regions in one figure**.
4. The last section contains extra, experimental plots.

**Independent of the reconstruction notebook:** this notebook needs no OGGM, only `numpy`, `pandas` and `matplotlib`. Edit the config (cell 2) and run all cells. Regions whose folder or file is missing are reported and skipped, so you can also run it while some regions are still unfinished.

In [ ]:
# ======================================================================
# 1. IMPORTS
# ======================================================================
import os
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter

In [ ]:
# ======================================================================
# 2. CONFIGURATION
# ======================================================================
# Which reconstruction to aggregate: "RGI60" (folders vas_rgi60_regionXX) or "RGI70" (vas_rgi70_regionXX)
RGI_VERSION = "RGI60"

# Folder that contains the region folders (same WORK_ROOT as the reconstruction notebook).
# On a server you can set the environment variable OGGM_WORK_ROOT instead of editing this line.
WORK_ROOT = Path(os.environ.get("OGGM_WORK_ROOT", "~/oggm_work")).expanduser()

# Where tables and figures are saved (separate folder per RGI version, so nothing is overwritten)
OUT_DIR = WORK_ROOT / f"global_aggregation_{RGI_VERSION.lower()}"

# Reconstruction period (must match the file names of the reconstruction run)
MODEL_START_YEAR = 1901
CRU_RECON_END_YEAR = 2019

# Region names
RGI_REGION_NAMES = {
    1: "Alaska", 2: "Western Canada and USA", 3: "Arctic Canada North",
    4: "Arctic Canada South", 5: "Greenland Periphery", 6: "Iceland",
    7: "Svalbard", 8: "Scandinavia", 9: "Russian Arctic", 10: "North Asia",
    11: "Central Europe", 12: "Caucasus and Middle East", 13: "Central Asia",
    14: "South Asia West", 15: "South Asia East", 16: "Low Latitudes",
    17: "Southern Andes", 18: "New Zealand", 19: "Antarctic and Subantarctic",
}

REGIONS = list(range(1, 20))      # which regions to read
GLOBAL_EXCLUDE = []               # regions left OUT of the global sum, e.g. [5, 19]
                                  # (peripheral glaciers of the ice sheets are often
                                  #  reported separately; [] = all 19 regions)

# Figure settings
SAVE_FORMATS = ["png", "pdf"]     # pdf is vector graphics: best for a thesis
DPI = 300
SHOW_PLOTS = True                 # False: only save, do not display
PLOT_STYLE_FONT_SIZE = 12

# Physical constants (same as the reconstruction notebook)
RHO_ICE = 900.0
RHO_WATER = 1000.0
OCEAN_AREA_KM2 = 3.625e8

# ----- derived (do not edit) -----
if RGI_VERSION not in ("RGI60", "RGI70"):
    raise ValueError('RGI_VERSION must be "RGI60" or "RGI70".')
RGI_LABEL = "RGI 6.0" if RGI_VERSION == "RGI60" else "RGI 7.0"
FOLDER_PREFIX = f"vas_{RGI_VERSION.lower()}_region"
HIST_FILE = f"regional_results_hist{MODEL_START_YEAR}_{CRU_RECON_END_YEAR}.csv"
OUT_DIR.mkdir(parents=True, exist_ok=True)
plt.rcParams.update({"font.size": PLOT_STYLE_FONT_SIZE})
print("RGI version:", RGI_LABEL)
print("Work root :", WORK_ROOT)
print("Output    :", OUT_DIR)
print("Input file:", HIST_FILE)

In [ ]:
# ======================================================================
# 3. LOAD ALL REGIONS
# ======================================================================
def region_file(region):
    return WORK_ROOT / f"{FOLDER_PREFIX}{region:02d}" / HIST_FILE


def add_mass_and_sle(df, volume_ref_km3):
    """Mass loss and sea-level equivalent relative to a reference volume."""
    df = df.copy()
    loss = volume_ref_km3 - df["volume_km3"]
    df["volume_loss_km3"] = loss
    df["mass_loss_Gt"] = loss * RHO_ICE / RHO_WATER            # 1 km3 water = 1 Gt
    df["sea_level_change_mm"] = df["mass_loss_Gt"] / OCEAN_AREA_KM2 * 1e6
    df["annual_mass_change_Gt_yr"] = df["mass_loss_Gt"].diff()  # positive = loss
    df["annual_sle_mm_yr"] = df["sea_level_change_mm"].diff()
    return df


def add_changes(df):
    """Changes relative to the first year, plus mass loss and sea level."""
    df = df.sort_values("year").reset_index(drop=True).copy()
    first = df.iloc[0]
    df["volume_change_km3"] = df["volume_km3"] - first["volume_km3"]
    df["area_change_km2"] = df["area_km2"] - first["area_km2"]
    df["length_change_km"] = df["length_km"] - first["length_km"]
    return add_mass_and_sle(df, float(first["volume_km3"]))


def load_all_regions():
    frames, missing = [], []
    for r in REGIONS:
        f = region_file(r)
        if not f.exists():
            missing.append(r)
            continue
        df = pd.read_csv(f)
        needed = {"year", "volume_km3", "area_km2"}
        if not needed.issubset(df.columns):
            print(f"Region {r:02d}: file lacks columns {needed - set(df.columns)}, skipped.")
            missing.append(r)
            continue
        if "length_km" not in df:
            df["length_km"] = np.nan
        df = add_changes(df[["year", "volume_km3", "area_km2", "length_km"]])
        df.insert(0, "region", r)
        df.insert(1, "region_name", RGI_REGION_NAMES[r])
        frames.append(df)
    if not frames:
        raise RuntimeError(f"No regional result files found under {WORK_ROOT}.")
    return pd.concat(frames, ignore_index=True), missing


regional_all, missing_regions = load_all_regions()
REGIONS_PRESENT = sorted(int(r) for r in regional_all["region"].unique())
by_region = {r: g.reset_index(drop=True) for r, g in regional_all.groupby("region")}

print(f"Loaded {len(REGIONS_PRESENT)} regions: {REGIONS_PRESENT}")
if missing_regions:
    print(f"MISSING (folder/file not found or unreadable): {missing_regions}")
    print("  -> the global totals below do NOT include these regions!")

year_ranges = regional_all.groupby("region")["year"].agg(["min", "max", "count"])
if year_ranges[["min", "max"]].drop_duplicates().shape[0] > 1:
    print("WARNING: regions have different year ranges:\n", year_ranges)

regional_all.to_csv(OUT_DIR / "all_regions_timeseries.csv", index=False)

In [ ]:
# ======================================================================
# 4. GLOBAL AGGREGATION
# ======================================================================
def make_global(exclude=()):
    regions = [r for r in REGIONS_PRESENT if r not in exclude]
    sub = regional_all[regional_all["region"].isin(regions)]
    g = (sub.groupby("year", as_index=False)
            .agg(volume_km3=("volume_km3", "sum"),
                 area_km2=("area_km2", "sum"),
                 length_km=("length_km", "sum"),
                 n_regions=("region", "nunique")))
    if (g["n_regions"] != len(regions)).any():
        print("WARNING: some years do not contain all regions; the global sum is incomplete there.")
    g = add_changes(g.drop(columns="n_regions"))
    return g, regions


global_df, GLOBAL_REGIONS = make_global(GLOBAL_EXCLUDE)
global_df.to_csv(OUT_DIR / "global_timeseries.csv", index=False)
print(f"Global = sum of {len(GLOBAL_REGIONS)} regions: {GLOBAL_REGIONS}")
if GLOBAL_EXCLUDE:
    print("Excluded from global:", GLOBAL_EXCLUDE)


# ----- summary table per region + global -----
def summary_row(df, label, region):
    a, b = df.iloc[0], df.iloc[-1]
    return {
        "region": region, "region_name": label,
        "start_year": int(a["year"]), "end_year": int(b["year"]),
        "volume_start_km3": a["volume_km3"], "volume_end_km3": b["volume_km3"],
        "volume_loss_km3": a["volume_km3"] - b["volume_km3"],
        "volume_loss_percent": 100 * (1 - b["volume_km3"] / a["volume_km3"]) if a["volume_km3"] > 0 else np.nan,
        "area_start_km2": a["area_km2"], "area_end_km2": b["area_km2"],
        "area_loss_percent": 100 * (1 - b["area_km2"] / a["area_km2"]) if a["area_km2"] > 0 else np.nan,
        "mass_loss_Gt": b["mass_loss_Gt"], "sea_level_mm": b["sea_level_change_mm"],
    }


rows = [summary_row(by_region[r], RGI_REGION_NAMES[r], r) for r in REGIONS_PRESENT]
summary = pd.DataFrame(rows)
summary["share_of_global_sle_percent"] = np.where(
    summary["region"].isin(GLOBAL_REGIONS),
    100 * summary["sea_level_mm"] / global_df["sea_level_change_mm"].iloc[-1], np.nan)
summary = pd.concat([summary, pd.DataFrame([summary_row(global_df, "Global", "global")])],
                    ignore_index=True)
summary.to_csv(OUT_DIR / "summary_regions_and_global.csv", index=False)

pd.set_option("display.width", 200)
print(summary[["region", "region_name", "volume_start_km3", "volume_end_km3",
               "volume_loss_percent", "mass_loss_Gt", "sea_level_mm"]]
      .round(2).to_string(index=False))

In [ ]:
# ======================================================================
# 5. PLOT HELPERS
# ======================================================================
# column, y label, file key, log scale useful for the all-regions plot?
VARS = [
    ("volume_km3", "Glacier volume (km$^3$)", "volume", True),
    ("area_km2", "Glacier area (km$^2$)", "area", True),
    ("sea_level_change_mm",
     f"Sea-level equivalent relative to {MODEL_START_YEAR} (mm)", "sea_level", False),
]

# one colour per region (tab20), dashed lines for regions 11-19 to tell similar colours apart
REGION_COLORS = {r: plt.cm.tab20((r - 1) % 20) for r in RGI_REGION_NAMES}
REGION_LS = {r: "-" if r <= 10 else "--" for r in RGI_REGION_NAMES}
GLOBAL_COLOR = "black"


def region_label(r):
    return f"{r:02d} {RGI_REGION_NAMES[r]}"


def save_fig(fig, name, subdir=None):
    folder = OUT_DIR if subdir is None else OUT_DIR / subdir
    folder.mkdir(parents=True, exist_ok=True)
    for ext in SAVE_FORMATS:
        fig.savefig(folder / f"{name}.{ext}", dpi=DPI, bbox_inches="tight")
    if SHOW_PLOTS:
        plt.show()
    plt.close(fig)


def thousands(x, _pos):
    return f"{x:,.0f}".replace(",", " ") if abs(x) >= 10000 else f"{x:g}"


def style_axis(ax, ylabel, zero_line=False):
    ax.set_xlabel("Year")
    ax.set_ylabel(ylabel)
    ax.set_xlim(MODEL_START_YEAR, CRU_RECON_END_YEAR)
    ax.grid(True, alpha=0.3)
    if zero_line:
        ax.axhline(0, color="black", linewidth=0.8)

## A. One plot per region and variable

In [ ]:
# ======================================================================
# 6. PER-REGION PLOTS (saved in OUT_DIR/per_region)
# ======================================================================
for r in REGIONS_PRESENT:
    d = by_region[r]
    for col, ylabel, key, _ in VARS:
        fig, ax = plt.subplots(figsize=(9, 5))
        ax.plot(d["year"], d[col], color=GLOBAL_COLOR, linewidth=2, label="VAS reconstruction (CRU)")
        style_axis(ax, ylabel, zero_line=(key == "sea_level"))
        ax.set_title(f"{RGI_VERSION}-{r:02d} - {RGI_REGION_NAMES[r]}\n"
                     f"VAS reconstruction {MODEL_START_YEAR}-{CRU_RECON_END_YEAR} (CRU-forced)")
        ax.legend()
        fig.tight_layout()
        save_fig(fig, f"region{r:02d}_{key}", subdir="per_region")
print(f"Saved {len(REGIONS_PRESENT) * len(VARS)} per-region plots in {OUT_DIR / 'per_region'}")

## B. Global plots

In [ ]:
# ======================================================================
# 7. GLOBAL PLOTS (saved in OUT_DIR)
# ======================================================================
excl_txt = f" (excl. regions {GLOBAL_EXCLUDE})" if GLOBAL_EXCLUDE else ""
for col, ylabel, key, _ in VARS:
    fig, ax = plt.subplots(figsize=(9, 5))
    ax.plot(global_df["year"], global_df[col], color=GLOBAL_COLOR, linewidth=2,
            label="VAS reconstruction (CRU)")
    style_axis(ax, ylabel, zero_line=(key == "sea_level"))
    ax.set_title(f"Global glaciers ({RGI_LABEL}){excl_txt}\n"
                 f"VAS reconstruction {MODEL_START_YEAR}-{CRU_RECON_END_YEAR} (CRU-forced)")
    ax.legend()
    fig.tight_layout()
    save_fig(fig, f"global_{key}")

## C. All regions in one plot (thesis figures)
Volume and area differ by orders of magnitude between regions, so both a linear and a logarithmic version are made. The sea-level plot also shows the global total (black, dashed).

In [ ]:
# ======================================================================
# 8. ALL REGIONS IN ONE FIGURE
# ======================================================================
def plot_all_regions(col, ylabel, key, logy=False, global_line=False, ylim=None):
    fig, ax = plt.subplots(figsize=(12, 7))
    for r in REGIONS_PRESENT:
        d = by_region[r]
        ax.plot(d["year"], d[col], color=REGION_COLORS[r], linestyle=REGION_LS[r],
                linewidth=1.8, label=region_label(r))
    if global_line:
        ax.plot(global_df["year"], global_df[col], color=GLOBAL_COLOR, linestyle=(0, (5, 2)),
                linewidth=2.8, label="Global" + (f" (excl. {GLOBAL_EXCLUDE})" if GLOBAL_EXCLUDE else ""))
    style_axis(ax, ylabel, zero_line=(key == "sea_level"))
    if logy:
        ax.set_yscale("log")
    if ylim:
        ax.set_ylim(*ylim)
    ax.set_title(f"All {RGI_LABEL} regions - VAS reconstruction {MODEL_START_YEAR}-{CRU_RECON_END_YEAR} (CRU-forced)")
    ax.legend(loc="center left", bbox_to_anchor=(1.01, 0.5), fontsize=10, frameon=False)
    fig.tight_layout()
    suffix = "_log" if logy else ""
    save_fig(fig, f"all_regions_{key}{suffix}")


for col, ylabel, key, log_ok in VARS:
    plot_all_regions(col, ylabel, key, logy=False, global_line=(key == "sea_level"))
    if log_ok:
        plot_all_regions(col, ylabel, key, logy=True)

In [ ]:
# ======================================================================
# 9. SMALL MULTIPLES: ALL REGIONS AS PANELS (own y-axis per panel)
# ======================================================================
def plot_grid(col, ylabel, key, ncols=4):
    n = len(REGIONS_PRESENT)
    nrows = int(np.ceil(n / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(3.6 * ncols, 2.7 * nrows), sharex=True)
    axes = np.atleast_1d(axes).ravel()
    for ax, r in zip(axes, REGIONS_PRESENT):
        d = by_region[r]
        ax.plot(d["year"], d[col], color=REGION_COLORS[r], linewidth=1.8)
        ax.set_title(region_label(r), fontsize=10)
        ax.set_xlim(MODEL_START_YEAR, CRU_RECON_END_YEAR)
        ax.grid(True, alpha=0.3)
        ax.tick_params(labelsize=9)
    for ax in axes[n:]:
        ax.axis("off")
    fig.supxlabel("Year")
    fig.supylabel(ylabel)
    fig.suptitle(f"VAS reconstruction {MODEL_START_YEAR}-{CRU_RECON_END_YEAR} per region (CRU-forced)", y=1.0)
    fig.tight_layout()
    save_fig(fig, f"all_regions_grid_{key}")


for col, ylabel, key, _ in VARS:
    plot_grid(col, ylabel, key)

## D. Experiments: extra plots that may be interesting
Delete or change whatever you do not need.

In [ ]:
# ======================================================================
# 10. EXPERIMENT 1: relative volume left (% of start volume), all regions
# ======================================================================
fig, ax = plt.subplots(figsize=(12, 7))
for r in REGIONS_PRESENT:
    d = by_region[r]
    ax.plot(d["year"], 100 * d["volume_km3"] / d["volume_km3"].iloc[0],
            color=REGION_COLORS[r], linestyle=REGION_LS[r], linewidth=1.8, label=region_label(r))
ax.plot(global_df["year"], 100 * global_df["volume_km3"] / global_df["volume_km3"].iloc[0],
        color=GLOBAL_COLOR, linestyle=(0, (5, 2)), linewidth=2.8, label="Global")
style_axis(ax, f"Glacier volume (% of {MODEL_START_YEAR})")
ax.set_title("Relative volume change per region")
ax.legend(loc="center left", bbox_to_anchor=(1.01, 0.5), fontsize=10, frameon=False)
fig.tight_layout()
save_fig(fig, "exp_relative_volume")

In [ ]:
# ======================================================================
# 11. EXPERIMENT 2: total sea-level contribution per region (bar chart)
# ======================================================================
reg_sum = summary[summary["region"] != "global"].copy()
reg_sum = reg_sum.sort_values("sea_level_mm", ascending=True)

fig, ax = plt.subplots(figsize=(9, 7))
colors = [REGION_COLORS[int(r)] for r in reg_sum["region"]]
bars = ax.barh([region_label(int(r)) for r in reg_sum["region"]], reg_sum["sea_level_mm"], color=colors)
for b, v in zip(bars, reg_sum["sea_level_mm"]):
    ax.text(b.get_width(), b.get_y() + b.get_height() / 2, f"  {v:.1f}", va="center", fontsize=9)
ax.set_xlabel(f"Sea-level equivalent {MODEL_START_YEAR}-{CRU_RECON_END_YEAR} (mm)")
ax.set_title("Contribution of each region to sea-level rise")
ax.grid(True, axis="x", alpha=0.3)
ax.set_xlim(0, reg_sum["sea_level_mm"].max() * 1.12)
fig.tight_layout()
save_fig(fig, "exp_sle_per_region_bar")

In [ ]:
# ======================================================================
# 12. EXPERIMENT 3: cumulative sea-level contribution, stacked by region
# ======================================================================
order = [r for r in reg_sum["region"].astype(int)][::-1]       # largest contributor at the bottom
years = global_df["year"].values
stack = [by_region[r].set_index("year").reindex(years)["sea_level_change_mm"].values for r in order]

fig, ax = plt.subplots(figsize=(12, 7))
ax.stackplot(years, stack, colors=[REGION_COLORS[r] for r in order],
             labels=[region_label(r) for r in order], alpha=0.95)
style_axis(ax, f"Cumulative sea-level equivalent since {MODEL_START_YEAR} (mm)")
ax.set_title("Cumulative sea-level contribution by region")
handles, labels = ax.get_legend_handles_labels()
ax.legend(handles[::-1], labels[::-1], loc="center left", bbox_to_anchor=(1.01, 0.5),
          fontsize=10, frameon=False)
fig.tight_layout()
save_fig(fig, "exp_sle_stacked")

In [ ]:
# ======================================================================
# 13. EXPERIMENT 4: global annual sea-level rate (with 10-year running mean)
# ======================================================================
g = global_df.copy()
g["rate_smooth"] = g["annual_sle_mm_yr"].rolling(10, center=True, min_periods=5).mean()

fig, ax = plt.subplots(figsize=(10, 5.5))
ax.bar(g["year"], g["annual_sle_mm_yr"], color="lightgrey", label="Annual")
ax.plot(g["year"], g["rate_smooth"], color=GLOBAL_COLOR, linewidth=2.5, label="10-year running mean")
style_axis(ax, "Global sea-level rate (mm yr$^{-1}$)", zero_line=True)
ax.set_title("Glacier contribution to sea-level rise: annual rate")
ax.legend()
fig.tight_layout()
save_fig(fig, "exp_global_sle_rate")

In [ ]:
# ======================================================================
# 14. EXPERIMENT 5: heatmap of sea-level rate per region and period (mm/yr)
# ======================================================================
edges = list(range(MODEL_START_YEAR, CRU_RECON_END_YEAR, 10)) + [CRU_RECON_END_YEAR]
periods = list(zip(edges[:-1], edges[1:]))
period_labels = [f"{a}-{b}" for a, b in periods]

heat = pd.DataFrame(index=[region_label(r) for r in REGIONS_PRESENT], columns=period_labels, dtype=float)
for r in REGIONS_PRESENT:
    s = by_region[r].set_index("year")["sea_level_change_mm"]
    for (a, b), lab in zip(periods, period_labels):
        if a in s.index and b in s.index:
            heat.loc[region_label(r), lab] = (s[b] - s[a]) / (b - a)

fig, ax = plt.subplots(figsize=(11, 7.5))
im = ax.imshow(heat.values.astype(float), aspect="auto", cmap="viridis")
ax.set_xticks(range(len(period_labels)))
ax.set_xticklabels(period_labels, rotation=45, ha="right")
ax.set_yticks(range(len(heat.index)))
ax.set_yticklabels(heat.index)
cb = fig.colorbar(im, ax=ax)
cb.set_label("Sea-level rate (mm yr$^{-1}$)")
ax.set_title("Glacier sea-level contribution per region and period")
fig.tight_layout()
save_fig(fig, "exp_sle_rate_heatmap")
heat.to_csv(OUT_DIR / "sle_rate_region_period.csv")

In [ ]:
# ======================================================================
# 15. EXPERIMENT 6: area vs. volume loss (% of start) per region
# ======================================================================
rs = summary[summary["region"] != "global"].copy()
fig, ax = plt.subplots(figsize=(8, 7))
size = 40 + 600 * rs["volume_start_km3"] / rs["volume_start_km3"].max()
ax.scatter(rs["area_loss_percent"], rs["volume_loss_percent"], s=size,
           c=[REGION_COLORS[int(r)] for r in rs["region"]], edgecolor="black", alpha=0.85)
for _, row in rs.iterrows():
    ax.annotate(f"{int(row['region']):02d}", (row["area_loss_percent"], row["volume_loss_percent"]),
                ha="center", va="center", fontsize=8)
lim = max(rs["area_loss_percent"].max(), rs["volume_loss_percent"].max()) * 1.05
ax.plot([0, lim], [0, lim], color="grey", linestyle=":", label="1:1")
ax.set_xlabel(f"Area lost {MODEL_START_YEAR}-{CRU_RECON_END_YEAR} (%)")
ax.set_ylabel(f"Volume lost {MODEL_START_YEAR}-{CRU_RECON_END_YEAR} (%)")
ax.set_title("Relative area vs. volume loss (bubble size = start volume)")
ax.grid(True, alpha=0.3)
ax.legend()
fig.tight_layout()
save_fig(fig, "exp_area_vs_volume_loss")

print("\nDone. Everything is saved in", OUT_DIR)